# M6.3 · torch.func: vmap, grad, jvp, jacrev

Fonksiyonel dönüşümlerle türev. Örnek başı gradyanları, Jacobian'ı, JVP/VJP'yi ve Hessian-vektör çarpımını elle hesaplıyor, saf Python'da dual sayılarla forward-mode AD yazıyor, NumPy'da lojistik regresyon için örnek başı gradyan ve Hessian'ı matris oluşturmadan çarpan HVP'yi kuruyor, PyTorch'ta `vmap(grad(…))`, `jacrev`/`jacfwd` ve `jvp(grad)` kullanıyor ve en sonda örnek başı kırpmalı DP-SGD adımını vmap ile yazıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m6-3-torch-func.html](https://m-gonul.github.io/pytorch-lab/m6-3-torch-func.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m6_3_torch_func.py
CHECKS = {
 "b": "import math\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n    D = need(ns, \"Dual\", \"sınıf\")\n\n    def ops():\n        a, b = D(2.0, 1.0), D(3.0, 0.5)\n        p = a * b + 2 * a - b * 1.5 + 1\n        close([p.val, p.dot], [6 + 4 - 4.5 + 1, 1 * 3 + 2 * 0.5 + 2 - 0.75])\n    R.test(\"Dual aritmetiği (sağdan sayılar dahil)\", ops)\n\n    def fns():\n        d = need(ns, \"dsin\")(D(0.5, 2.0)); e = need(ns, \"dexp\")(D(0.5, 2.0))\n        close([d.val, d.dot, e.val, e.dot], [math.sin(0.5), 2 * math.cos(0.5), math.exp(0.5), 2 * math.exp(0.5)])\n    R.test(\"dsin, dexp\", fns)\n    R.test(\"jvp (elle örnek)\", lambda: close(need(ns, \"jvp\")(lambda x: [x[0] * x[1], x[0] * x[0]], [1.0, 2.0], [1.0, 1.0]), [[2, 1], [3, 2]]))\n\n    def jac():\n        sin, exp = ns[\"dsin\"], ns[\"dexp\"]\n        f = lambda x: [sin(x[0] * x[1]) + x[2], exp(x[0] - x[2]) * x[1]]\n        x = [0.3, -1.2, 0.7]\n        J = need(ns, \"jacobian_fwd\")(f, x)\n        g = lambda x: [math.sin(x[0] * x[1]) + x[2], math.exp(x[0] - x[2]) * x[1]]\n        h = 1e-6\n        num = [[(g(x[:j] + [x[j] + h] + x[j + 1:])[i] - g(x[:j] + [x[j] - h] + x[j + 1:])[i]) / (2 * h) for j in range(3)] for i in range(2)]\n        close(J, num, atol=1e-6)\n    R.test(\"jacobian_fwd = sayısal Jacobian (2 × 3)\", jac)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(0)\n    X, y, w = rng.normal(size=(40, 5)), rng.integers(0, 2, 40), rng.normal(size=5)\n    loss_i = lambda w, i: -(y[i] * np.log(1 / (1 + np.exp(-X[i] @ w))) + (1 - y[i]) * np.log(1 - 1 / (1 + np.exp(-X[i] @ w))))\n    loss = lambda w: np.mean([loss_i(w, i) for i in range(40)])\n\n    def psg():\n        G = need(ns, \"per_sample_grads\")(w, X, y)\n        h = 1e-6\n        for i in (0, 7, 39):\n            num = [(loss_i(w + h * e, i) - loss_i(w - h * e, i)) / (2 * h) for e in np.eye(5)]\n            close(G[i], num, atol=1e-6, what=f\"örnek {i}\")\n    R.test(\"per_sample_grads = sayısal (örnek başı)\", psg)\n\n    def hs():\n        H = need(ns, \"hessian\")(w, X)\n        h = 1e-4\n        g = lambda w: need(ns, \"per_sample_grads\")(w, X, y).mean(0)\n        num = np.stack([(g(w + h * e) - g(w - h * e)) / (2 * h) for e in np.eye(5)], 1)\n        close(H, num, atol=1e-6)\n        ensure(np.all(np.linalg.eigvalsh(H) >= -1e-12), \"Hessian PSD olmalı (konveks kayıp)\")\n    R.test(\"hessian = gradyanın Jacobian'ı, PSD\", hs)\n\n    def hv():\n        v = rng.normal(size=5)\n        close(need(ns, \"hvp\")(w, X, v), ns[\"hessian\"](w, X) @ v)\n    R.test(\"hvp = H v (H kurulmadan)\", hv)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    torch.manual_seed(0)\n    model = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 3)).double()\n    x = torch.randn(6, 4, dtype=torch.float64); y = torch.randint(0, 3, (6,))\n\n    def psg():\n        G = need(ns, \"per_sample_grads\")(model, x, y)\n        ensure(set(G) == {k for k, _ in model.named_parameters()}, \"parametre adlarıyla sözlük\")\n        for i in range(6):\n            model.zero_grad()\n            F.cross_entropy(model(x[i:i + 1]), y[i:i + 1]).backward()\n            for k, p in model.named_parameters():\n                close(G[k][i], p.grad, what=f\"örnek {i}, {k}\")\n    R.test(\"per_sample_grads = örnek başı backward\", psg)\n\n    def jac():\n        W = torch.randn(3, 5, dtype=torch.float64)\n        f = lambda v: torch.tanh(W @ v) * v[:3]\n        v = torch.randn(5, dtype=torch.float64)\n        Jr, Jf = need(ns, \"jacobians\")(f, v)\n        ref = torch.autograd.functional.jacobian(f, v)\n        close(Jr, ref, what=\"jacrev\"); close(Jf, ref, what=\"jacfwd\")\n        ensure(tuple(Jr.shape) == (3, 5), \"Jacobian (3, 5)\")\n    R.test(\"jacrev = jacfwd = autograd.functional.jacobian\", jac)\n\n    def hv():\n        A = torch.randn(6, 6, dtype=torch.float64)\n        f = lambda v: (torch.sin(A @ v) ** 2).sum() + (v ** 4).sum()\n        x0, v = torch.randn(6, dtype=torch.float64), torch.randn(6, dtype=torch.float64)\n        H = torch.autograd.functional.hessian(f, x0)\n        close(need(ns, \"hvp\")(f, x0, v), H @ v)\n    R.test(\"hvp = H v\", hv)\n",
 "e": "import copy\nimport torch\nimport torch.nn.functional as F\nfrom torch.func import functional_call, vmap, grad\nfrom labcheck import close, need, ensure, bench, fmt_time, uses_any\n\n\ndef ref_step(model, x, y, C, lr):\n    B = x.shape[0]\n    grads = []\n    for i in range(B):\n        model.zero_grad()\n        F.cross_entropy(model(x[i:i + 1]), y[i:i + 1]).backward()\n        g = [p.grad.clone() for p in model.parameters()]\n        n = torch.sqrt(sum(t.pow(2).sum() for t in g))\n        grads.append([t * min(1.0, C / (n.item() + 1e-12)) for t in g])\n    with torch.no_grad():\n        for j, p in enumerate(model.parameters()):\n            p -= lr * sum(g[j] for g in grads) / B\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"vmap kullanılıyor\", lambda: ensure(uses_any(src, [\"vmap\"]), \"örnek başı gradyan için vmap kullan\"))\n    torch.manual_seed(0)\n    base = torch.nn.Sequential(torch.nn.Linear(10, 16), torch.nn.Tanh(), torch.nn.Linear(16, 3)).double()\n    x = torch.randn(12, 10, dtype=torch.float64); y = torch.randint(0, 3, (12,))\n\n    def noiseless():\n        a, b = copy.deepcopy(base), copy.deepcopy(base)\n        need(ns, \"dp_sgd_step\")(a, x, y, max_norm=0.3, noise_mult=0.0, lr=0.1)\n        ref_step(b, x, y, 0.3, 0.1)\n        for p, q in zip(a.parameters(), b.parameters()):\n            close(p, q)\n    R.test(\"gürültüsüz adım = örnek başı kırpmalı döngü\", noiseless)\n\n    def noise():\n        a, b = copy.deepcopy(base), copy.deepcopy(base)\n        need(ns, \"dp_sgd_step\")(a, x, y, max_norm=0.3, noise_mult=0.0, lr=0.1)\n        g = torch.Generator().manual_seed(0)\n        ns[\"dp_sgd_step\"](b, x, y, max_norm=0.3, noise_mult=2.0, lr=0.1, generator=g)\n        diff = torch.cat([(p - q).flatten() for p, q in zip(a.parameters(), b.parameters())])\n        want = 0.1 * 2.0 * 0.3 / 12\n        ensure(abs(diff.std().item() - want) < 0.25 * want, f\"gürültünün std'si {diff.std().item():.2e}, beklenen ≈ {want:.2e}\")\n    R.test(\"gürültü ölçeği σ·C·lr/B\", noise)\n\n    def clip():\n        a = copy.deepcopy(base)\n        before = [p.clone() for p in a.parameters()]\n        need(ns, \"dp_sgd_step\")(a, x * 100, y, max_norm=0.01, noise_mult=0.0, lr=1.0)\n        delta = torch.sqrt(sum((p - q).pow(2).sum() for p, q in zip(a.parameters(), before)))\n        ensure(delta <= 0.01 + 1e-9, f\"tüm örnekler kırpılınca adım ≤ C·lr olmalı ({delta.item():.4f})\")\n    R.test(\"kırpma: adım normu ≤ C·lr\", clip)\n    if \"dp_sgd_step\" not in ns:\n        return\n    m = torch.nn.Sequential(torch.nn.Linear(64, 256), torch.nn.ReLU(), torch.nn.Linear(256, 10))\n    xb, yb = torch.randn(128, 64), torch.randint(0, 10, (128,))\n    t_loop = bench(lambda: ref_step(m, xb, yb, 1.0, 0.0), repeat=1, number=1)\n    t_vm = bench(lambda: ns[\"dp_sgd_step\"](m, xb, yb, 1.0, 0.0, 0.0), repeat=3)\n    R.table(\"DP-SGD adımı, B = 128\", [\"yöntem\", \"süre\"], [[\"örnek başı backward döngüsü\", fmt_time(t_loop)], [\"senin vmap(grad)\", fmt_time(t_vm)]])\n"
}

In [ ]:
import importlib, labcheck, checks_m6_3_torch_func as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

`torch.func` (eski adıyla functorch) türevi **fonksiyon dönüşümü** olarak sunar:

| dönüşüm | ne verir | maliyet |
|---|---|---|
| `grad(f)` | $\nabla f$ (skaler $f$) | 1 geri geçiş |
| `vjp(f, x)` | $u\mapsto u^\top J$ | 1 geri geçiş |
| `jvp(f, (x,), (v,))` | $Jv$ (jvp) | 1 ileri geçiş (dual sayılar) |
| `jacrev` / `jacfwd` | $J$ | çıktı sayısı / girdi sayısı kadar |
| `vmap(f)` | $f$'yi batch boyutu üzerinde | tek batch'li çağrı |

Bunlar bileşebilir: `vmap(grad(f))` örnek başı gradyan, `jacfwd(jacrev(f))` Hessian, `jvp(grad(f), …)` Hessian-vektör çarpımı.

Bir `nn.Module`'ü parametrelerinin saf fonksiyonu olarak çağırmak için `torch.func.functional_call(model, params, (x,))`.

**Forward-mode** dual sayılarla çalışır: her değer $(a, \dot a)$ çifti, kurallar $(a,\dot a)\cdot(b,\dot b)=(ab,\ \dot ab+a\dot b)$ gibi.

## 1 · Elle

Ortalama gradyan 0 olduğu hâlde iki örneğin gradyan normu 4: toplu gradyana bakan bir yöntem, tek bir örneğin modele ne kadar etki ettiğini göremez. DP-SGD her örneğin gradyanını ayrı ayrı kırpar ki hiçbir örnek sınırsız etki bırakamasın.

JVP bir **sütun** kombinasyonu ($Jv$), VJP bir **satır** kombinasyonu ($u^\top J$).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**$\ell_i(w)=(wx_i-y_i)^2$, $w=1$, örnekler $(x,y)=(1,3),(2,1)$**

- `g1`: Örnek 1'in gradyanı
- `g2`: Örnek 2'nin gradyanı
- `gm`: Ortalama gradyan

**$f(x_1,x_2)=(x_1x_2,\ x_1^2)$, $x=(1,2)$**

- `jvp1`: JVP $J v$, $v=(1,1)$: 1. bileşen
- `vjp1`: VJP $u^\top J$, $u=(1,1)$: 1. bileşen
- `hvp1`: $g(x)=x_1^2x_2$ için $Hv$, $v=(1,0)$: 1. bileşen

In [ ]:
cevaplar = dict(
    g1=None,
    g2=None,
    gm=None,
    jvp1=None,
    vjp1=None,
    hvp1=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"g1":[-4,0],"g2":[4,0],"gm":[0,0],"jvp1":[3,0],"vjp1":[4,0],"hvp1":[4,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, forward-mode AD.

- `Dual(val, dot)` — `+`, `-`, `*` (sağdan sayılarla da: `__radd__`, `__rmul__`), ve modül fonksiyonları `dsin(d)`, `dexp(d)`.
- `jvp(f, x, v)` — `f`: `Dual` listesi → `Dual` listesi; `x`, `v` sayı listesi → `(f(x) değerleri, Jv)`.
- `jacobian_fwd(f, x)` — $n$ JVP ile ($v=e_j$), $m\times n$ liste.

In [ ]:
%%writefile stage_b.py
import math


class Dual:
    def __init__(self, val, dot=0.0):
        self.val, self.dot = val, dot

    def __add__(self, o):
        ...

    __radd__ = __add__

    def __sub__(self, o):
        ...

    def __mul__(self, o):
        ...

    __rmul__ = __mul__


def dsin(d):
    ...


def dexp(d):
    ...


def jvp(f, x, v):
    ...


def jacobian_fwd(f, x):
    ...


f = lambda x: [x[0] * x[1], x[0] * x[0]]
print(jvp(f, [1.0, 2.0], [1.0, 1.0]))    # ([2, 1], [3, 2])
print(jacobian_fwd(f, [1.0, 2.0]))       # [[2, 1], [2, 0]]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

NumPy, lojistik regresyon $p=\sigma(Xw)$, $\ \ell=\frac1N\sum_i\text{BCE}(p_i,y_i)$. **Döngü yok.**

- `per_sample_grads(w, X, y)` → `(N, d)`: $(p_i-y_i)x_i$.
- `hessian(w, X)` → $\frac1NX^\top\operatorname{diag}(p(1-p))X$.
- `hvp(w, X, v)` → $Hv$ ama $H$'yi kurmadan: $\frac1NX^\top\big(p(1-p)\odot Xv\big)$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def per_sample_grads(w, X, y):
    ...


def hessian(w, X):
    ...


def hvp(w, X, v):
    ...


rng = np.random.default_rng(0)
X, y, w = rng.normal(size=(6, 3)), rng.integers(0, 2, 6), rng.normal(size=3)
print(per_sample_grads(w, X, y).mean(0))
print(hvp(w, X, np.ones(3)), hessian(w, X) @ np.ones(3))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `per_sample_grads(model, x, y)` — `{ad: (B, …)}` sözlüğü. `params = {k: v.detach() for k, v in model.named_parameters()}`, tek örneklik kayıp `functional_call(model, params, (x[None],))` ile, sonra `vmap(grad(loss_one), in_dims=(None, 0, 0))`.
- `jacobians(f, x)` → `(jacrev(f)(x), jacfwd(f)(x))`.
- `hvp(f, x, v)` — `jvp(grad(f), (x,), (v,))[1]`.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch.func import functional_call, vmap, grad, jacrev, jacfwd, jvp


def per_sample_grads(model, x, y):
    ...


def jacobians(f, x):
    ...


def hvp(f, x, v):
    ...


torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.Tanh(), torch.nn.Linear(8, 3))
g = per_sample_grads(model, torch.randn(5, 4), torch.randint(0, 3, (5,)))
print({k: tuple(v.shape) for k, v in g.items()})

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

`dp_sgd_step(model, x, y, max_norm, noise_mult, lr, generator=None)` — bir DP-SGD adımı:

1. Örnek başı gradyanlar (`vmap(grad(…))`).
2. Her örneğin **tüm parametreler üzerinden** gradyan normu $\lVert g_i\rVert$; ölçek $\min(1, C/\lVert g_i\rVert)$.
3. Kırpılmış gradyanları topla, $\mathcal N(0,\sigma^2C^2)$ gürültü ekle (`torch.randn(…, generator=generator)`), $B$'ye böl.
4. `p -= lr * g` (`torch.no_grad()` içinde).

Örnekler üzerinde döngü yok.

In [ ]:
%%writefile stage_e.py
import torch
import torch.nn.functional as F
from torch.func import functional_call, vmap, grad


def dp_sgd_step(model, x, y, max_norm, noise_mult, lr, generator=None):
    """Örnekler üzerinde döngü yok."""
    ...


torch.manual_seed(0)
model = torch.nn.Sequential(torch.nn.Linear(10, 32), torch.nn.ReLU(), torch.nn.Linear(32, 2))
x, y = torch.randn(64, 10), torch.randint(0, 2, (64,))
for _ in range(100):
    dp_sgd_step(model, x, y, max_norm=1.0, noise_mult=0.5, lr=0.5)
print(F.cross_entropy(model(x), y).item())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Finn, Abbeel, Levine (2017): MAML

[Finn, Abbeel, Levine — Model-Agnostic Meta-Learning for Fast Adaptation of Deep Networks (ICML 2017)](https://arxiv.org/abs/1703.03400)

MAML'ı torch.func ile yaz: functional_call ile parametreleri sözlük olarak geçir, iç döngüde grad ile adaptasyon, dış döngüde ikinci dereceden gradyan; görevler üzerinde vmap ile paralelleştir. Makalenin sinüs regresyonu deneyini (Şekil 2–3) tekrarla ve birinci dereceden yaklaşımı (FOMAML) karşılaştır.


**Adımlar**

1. Görev batch'ini vmap(in_dims) ile tek çağrıda işle.

**Kabul ölçütleri**

- K = 5 örnekle adaptasyon, ön eğitimli taban çizgisini belirgin geçer.
- MAML vs FOMAML farkı ölçülmüş.

## Makale ödevi: Abadi ve ark. (2016): diferansiyel gizlilikli derin öğrenme

[Abadi ve ark. — Deep Learning with Differential Privacy (CCS 2016)](https://arxiv.org/abs/1607.00133)

DP-SGD'yi (Algoritma 1) örnek başı gradyanlarla yaz: vmap(grad(...)) ile her örneğin gradyanını hesapla, normunu C'ye kırp, toplayıp N(0, σ²C²) gürültü ekle. MNIST'te gizlilik bütçesine (ε) göre doğruluk eğrisini çiz; örnek başı gradyanları for döngüsü ve Opacus benzeri hook'larla hız açısından karşılaştır.


**Adımlar**

1. ε hesabı için momentler muhasebesini (ya da RDP) bir kütüphaneyle doğrula.

**Kabul ölçütleri**

- vmap'li örnek başı gradyanlar döngüyle aynı, çok daha hızlı.
- ε–doğruluk ödünleşim grafiği.

## Kaynaklar

- [PyTorch Docs — torch.func (functorch) ve tutorial'ları: per-sample gradients, Jacobians/Hessians](https://docs.pytorch.org/docs/stable/func.html) — BSD-3-Clause
- [JAX — The Autodiff Cookbook](https://docs.jax.dev/en/latest/notebooks/autodiff_cookbook.html) — Apache-2.0
- [Baydin ve ark. — Automatic Differentiation in Machine Learning: a Survey](https://arxiv.org/abs/1502.05767) — arXiv
- [Abadi ve ark. — Deep Learning with Differential Privacy (DP-SGD)](https://arxiv.org/abs/1607.00133) — arXiv